In [ ]:
import pandas as pd
import numpy as np
import os
import json

import optuna
from optuna import Trial
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
import kaggle

os.environ['KAGGLE_USERNAME'] = json.load(open('/home/osman/.config/kaggle/kaggle.json'))['username']
os.environ['KAGGLE_KEY'] = json.load(open('/home/osman/.config/kaggle/kaggle.json'))['key']


In [2]:
train_data = pd.read_csv("train_data.csv")
test_data = pd.read_csv("test_data.csv")
sample_submission = pd.read_csv("sample_submission.csv")

In [ ]:
def recall_at_k(y_true, y_prob, k=0.1):
    """
    Tahmin edilen olasılıkların en üst k%'sını pozitif etiketleyerek recall değerini hesaplar.

    Parametreler:
        y_true (list): Gerçek ikili etiketler.
        y_prob (list): Tahmin edilen olasılıklar.
        k (float): Pozitif etiketlenecek olasılıkların yüzdelik dilimi (varsayılan 0.1).

    Döndürür:
        float: En iyi k% tahminlerindeki recall oranı.
    """
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    n = len(y_true)
    m = max(1, int(np.round(k * n)))
    order = np.argsort(-y_prob, kind="mergesort")
    top = order[:m]

    tp_at_k = y_true[top].sum()
    P = y_true.sum()

    return float(tp_at_k / P) if P > 0 else 0.0


def lift_at_k(y_true, y_prob, k=0.1):
    """
    Tahmin edilen olasılıkların en üst k%'sını pozitif etiketleyerek lift (precision/prevalence) değerini hesaplar.

    Parametreler:
        y_true (list): Gerçek ikili etiketler.
        y_prob (list): Tahmin edilen olasılıklar.
        k (float): Pozitif etiketlenecek olasılıkların yüzdelik dilimi (varsayılan 0.1).

    Döndürür:
        float: En iyi k% tahminlerindeki lift değeri.
    """
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    n = len(y_true)
    m = max(1, int(np.round(k * n)))
    order = np.argsort(-y_prob, kind="mergesort")
    top = order[:m]

    tp_at_k = y_true[top].sum()
    precision_at_k = tp_at_k / m
    prevalence = y_true.mean()

    return float(precision_at_k / prevalence) if prevalence > 0 else 0.0


def convert_auc_to_gini(auc):
    """
    ROC AUC skorunu Gini katsayısına dönüştürür.

    Gini katsayısı, ROC AUC skorunun doğrusal bir dönüşümüdür.

    Parametreler:
        auc (float): ROC AUC skoru (0 ile 1 arasında).

    Döndürür:
        float: Gini katsayısı (-1 ile 1 arasında).
    """
    return 2 * auc - 1


def ing_hubs_datathon_metric(y_true, y_prob):
    """
    Gini, recall@10% ve lift@10% metriklerini birleştiren özel bir metrik hesaplar.

    Metrik, her bir skoru bir baseline modelin metrik değerlerine göre oranlar ve aşağıdaki ağırlıkları uygular:
    - Gini: %40
    - Recall@10%: %30
    - Lift@10%: %30

    Parametreler:
        y_true (list): Gerçek ikili etiketler.
        y_prob (list): Tahmin edilen olasılıklar.

    Döndürür:
        float: Ağırlıklandırılmış bileşik skor.
    """
    # final metrik için ağırlıklar
    score_weights = {
        "gini": 0.4,
        "recall_at_10perc": 0.3,
        "lift_at_10perc": 0.3,
    }

    # baseline modelin her bir metrik için değerleri
    baseline_scores = {
        "roc_auc": 0.6925726757936908,
        "recall_at_10perc": 0.18469015795868773,
        "lift_at_10perc": 1.847159286784029,
    }

    # y_prob tahminleri için metriklerin hesaplanması
    roc_auc = roc_auc_score(y_true, y_prob)
    recall_at_10perc = recall_at_k(y_true, y_prob, k=0.1)
    lift_at_10perc = lift_at_k(y_true, y_prob, k=0.1)

    new_scores = {
        "roc_auc": roc_auc,
        "recall_at_10perc": recall_at_10perc,
        "lift_at_10perc": lift_at_10perc,
    }

    # roc auc değerlerinin gini değerine dönüştürülmesi
    baseline_scores["gini"] = convert_auc_to_gini(baseline_scores["roc_auc"])
    new_scores["gini"] = convert_auc_to_gini(new_scores["roc_auc"])

    # baseline modeline oranlama
    final_gini_score = new_scores["gini"] / baseline_scores["gini"]
    final_recall_score = new_scores["recall_at_10perc"] / baseline_scores["recall_at_10perc"]
    final_lift_score = new_scores["lift_at_10perc"] / baseline_scores["lift_at_10perc"]

    # ağırlıklandırılmış metriğin hesaplanması
    final_score = (
        final_gini_score * score_weights["gini"] +
        final_recall_score * score_weights["recall_at_10perc"] + 
        final_lift_score * score_weights["lift_at_10perc"]
    )
    return final_score

In [4]:
train_data

,age,tenure,cust_age_month,uses_mobile_eft,uses_cc,uses_any_digital_channel,mobile_eft_cnt_mean,mobile_eft_cnt_std,mobile_eft_cnt_min,mobile_eft_cnt_max,...,work_type_Unemployed,work_sector_Finance,work_sector_Healthcare,work_sector_Manufacturing,work_sector_Public Sector,work_sector_Retail,work_sector_Retired,work_sector_Student,work_sector_Technology,work_sector_Unemployed
0,64,135,633,1,0,1,2.238095,1.220851,1.0,5.0,...,0,0,0,0,0,0,0,0,1,0
1,22,47,217,1,1,1,1.676471,1.006662,1.0,4.0,...,0,0,0,0,0,0,0,1,0,0
2,27,108,216,1,1,1,2.555556,1.476309,1.0,6.0,...,0,1,0,0,0,0,0,0,0,0
3,40,187,293,1,1,1,7.142857,3.307839,4.0,14.0,...,1,0,0,0,0,0,0,0,0,1
4,64,218,550,1,1,1,0.793103,1.372675,0.0,5.0,...,0,0,0,0,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
133282,54,217,431,1,1,1,1.393939,1.657170,0.0,6.0,...,0,0,0,0,1,0,0,0,0,0
133283,47,37,527,1,1,1,2.000000,1.174440,1.0,5.0,...,0,0,0,0,1,0,0,0,0,0
133284,66,227,565,1,1,1,9.055556,5.796277,1.0,22.0,...,0,0,0,0,0,0,1,0,0,0
133285,31,156,216,1,1,1,3.576923,1.836803,1.0,7.0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
X = train_data.drop("churn", axis=1)
y = train_data["churn"]

In [ ]:
def objective(trial: Trial, X: pd.DataFrame, y: np.ndarray, n_splits: int = 5) -> float:
    # Hyperparametreler
    params = {
        'objective': 'binary',
        'verbose': False,
        'metric': 'binary_logloss',
        'verbosity': -1,
        'random_state': 42,
        'boosting_type': 'gbdt',
        'num_leaves': trial.suggest_int('num_leaves', 30, 200),
        'max_depth': trial.suggest_int('max_depth', 5, 15),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.1, 10.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.1, 10.0),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'subsample_freq': trial.suggest_int('subsample_freq', 1, 10),
    }

    # Dengesizlik: Pozitif sınıf ağırlığını optimize et
    neg, pos = np.bincount(y)
    scale_pos_weight = neg / pos
    params['scale_pos_weight'] = trial.suggest_float('scale_pos_weight', scale_pos_weight * 0.5, scale_pos_weight * 2)

    # K-Fold
    kf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    custom_scores = []

    for train_idx, val_idx in kf.split(X, y):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]
        

        model = lgb.LGBMClassifier(**params)
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)]
        )

        y_pred_proba = model.predict_proba(X_val)[:, 1]

        # Özel metrik
        score = ing_hubs_datathon_metric(y_val, y_pred_proba)
        custom_scores.append(score)

    return np.mean(custom_scores)

In [ ]:
# Optimize et
study = optuna.create_study(direction='maximize', study_name='lgbm-churn-ing-metric')
study.optimize(
    lambda trial: objective(trial, X, y),
    n_trials=50,
    show_progress_bar=True
)

print("Best trial score:", study.best_trial.value)
print("Best params:")
for key, value in study.best_trial.params.items():
    print(f"  {key}: {value}")

In [ ]:
# En iyi parametreler
best_params = study.best_trial.params.copy()
best_params['objective'] = 'binary'
best_params['random_state'] = 42

# Final model
final_model = lgb.LGBMClassifier(**best_params)
final_model.fit(X, y)

# Tahmin
y_pred_proba = final_model.predict_proba(X)
# Tüm metrikleri yazdır
gini = convert_auc_to_gini(roc_auc_score(y, y_pred_proba[:, 1]))
recall_10 = recall_at_k(y, y_pred_proba, k=0.1)
lift_10 = lift_at_k(y, y_pred_proba, k=0.1)
final_score = ing_hubs_datathon_metric(y, y_pred_proba[:, 1])

print("\n📊 FINAL MODEL SKORLARI (TÜM VERİ ÜZERİNDE):")
print(f"Gini:              {gini:.4f}")
print(f"Recall@10%:        {recall_10:.4f}")
print(f"Lift@10%:          {lift_10:.4f}")
print(f"Final ING Metric:  {final_score:.4f}")

In [ ]:
cv_scores = []
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
models:list[lgb.LGBMClassifier] = []
for train_idx, val_idx in kf.split(X, y):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    
    model = lgb.LGBMClassifier(**best_params)
    model.fit(X_train, y_train)
    
    y_pred_proba = model.predict_proba(X_val)[:, 1]
    score = ing_hubs_datathon_metric(y_val, y_pred_proba)
    cv_scores.append(score)
    models.append(model)

print(f"\n✅ 5-Fold CV ING Metric: {np.mean(cv_scores):.4f} ± {np.std(cv_scores):.4f}")

In [ ]:
test_predictions = np.mean([m.predict_proba(test_data)[:,1] for m in models], axis=0)

In [ ]:
sample_submission["churn"] = test_predictions

In [ ]:
sample_submission.to_csv('/tmp/submission.csv', index=False)
kaggle.api.competition_submit(
    file_name='/tmp/submission.csv', 
    message='lgbm with Optuna kfold and feature engineering history data ensemble kfold models', 
    competition='ing-hubs-turkiye-datathon'
)